# 13 — Cold start, diversidade e popularidade

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 13/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/13_cold_start_diversidade.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_13_cold_start_diversidade.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Identificar cold start
- Medir diversidade simples de gêneros e filtrar vistos

## Fundamentação conceitual
O problema de cold start ocorre quando não há histórico suficiente. Alternativas incluem popularidade com regularização, informações de conteúdo e onboarding. Diversidade e novidade são dimensões adicionais à acurácia.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Fallback sem histórico

In [2]:
st=ratings.groupby('movieId').rating.agg(['mean','count']);st['score']=(st['count']*st['mean']+40*ratings.rating.mean())/(st['count']+40)
def recommend_cold(k=10):return movies.set_index('movieId').loc[st.sort_values('score',ascending=False).head(k).index][['title','genres']]
print(recommend_cold(8).to_string())

                                             title                       genres
movieId                                                                        
318               Shawshank Redemption, The (1994)                  Crime|Drama
2959                             Fight Club (1999)  Action|Crime|Drama|Thriller
858                          Godfather, The (1972)                  Crime|Drama
260      Star Wars: Episode IV - A New Hope (1977)      Action|Adventure|Sci-Fi
50                      Usual Suspects, The (1995)       Crime|Mystery|Thriller
296                            Pulp Fiction (1994)  Comedy|Crime|Drama|Thriller
527                        Schindler's List (1993)                    Drama|War
2571                            Matrix, The (1999)       Action|Sci-Fi|Thriller


**Interpretação.** Uma recomendação inicial deve ser marcada como não personalizada.

## Diversidade de gêneros

In [3]:
titles=recommend_cold(10)
all_genres=set(g for row in titles.genres for g in row.split('|') if g!='(no genres listed)')
print('Gêneros distintos nos top 10:',len(all_genres),sorted(all_genres))

Gêneros distintos nos top 10: 10 ['Action', 'Adventure', 'Comedy', 'Crime', 'Drama', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War']


**Interpretação.** A diversidade medida por gênero é uma aproximação grosseira.

## Histórico curto

In [4]:
few=ratings.userId.value_counts().tail(5)
print('Usuários de menor atividade:',few.to_dict())

Usuários de menor atividade: {431: 20, 442: 20, 569: 20, 576: 20, 595: 20}


**Interpretação.** Poucos registros favorecem modelos regulares e fallback robusto.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** cobertura e diversidade. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **cobertura e diversidade** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **cobertura e diversidade** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Diversidade de gêneros não é diversidade intra-lista

Contar gêneros únicos é um **indicador bruto de variedade**, não uma medida de distância entre itens nem garantia de novidade, cobertura ou serendipidade. Uma alternativa é a distância média entre pares de itens (por exemplo, Jaccard nos conjuntos de gêneros). Avalie o trade-off com relevância e exposição.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Diversidade intra-lista ilustrativa: distância de Jaccard dos gêneros
from itertools import combinations
sets = [set(g.split('|')) - {'(no genres listed)'} for g in titles.genres]
dists = [1-len(a&b)/len(a|b) for a,b in combinations(sets,2) if (a|b)]
print('Distância média de Jaccard:', round(float(np.mean(dists)),3) if dists else 'indefinida')

Distância média de Jaccard: 0.768


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Diversidade não deve ser aplicada indiscriminadamente; explicite trade-offs entre relevância e exploração.

## Exercícios
Proponha uma etapa de onboarding e uma métrica de diversidade mais sensível.

## Soluções comentadas
Solicitar 3 gêneros favoritos é um onboarding simples; distância intra-lista pode medir diversidade par a par.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"k": len(titles), "n_generos_distintos": len(all_genres),
 "distancia_jaccard_media": float(np.mean(dists)) if dists else np.nan,
 "nota": "Exemplo didatico; nao e avaliacao de qualidade fora da amostra"}]).to_csv(
    out / "13_diversidade_topk.csv", index=False)
print('Outputs do capítulo 13 salvos em:', out.resolve())


Outputs do capítulo 13 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
